In [ ]:
"""
Download dos contratos do PNCP por intervalo de datas.
Lógica portada do notebook consulta_contratos.ipynb.
"""
from __future__ import annotations

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path
from typing import Callable, Optional

import pandas as pd
import requests

from app.utils.config import CONTRATOS_DIR

BASE_URL = "https://pncp.gov.br/api/consulta/v1/contratos"
TAMANHO_PAGINA = 500
MAX_WORKERS = 10
TENTATIVAS = 3
TIMEOUT = 30

Progresso = Callable[[float, str], None]


class ErroPNCP(RuntimeError):
    """Falha ao consultar o PNCP."""


def _consultar_pagina(sessao: requests.Session, params: dict) -> Optional[dict]:
    """Retorna o JSON da página ou None se todas as tentativas falharem."""
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resposta = sessao.get(BASE_URL, params=params, timeout=TIMEOUT)
            if resposta.status_code == 200:
                return resposta.json()
            if resposta.status_code == 204:  # sem conteúdo
                return {"data": [], "totalPaginas": 0}
        except (requests.RequestException, ValueError):
            pass
        time.sleep(tentativa)
    return None


def nome_arquivo(data_inicial: date, data_final: date, uf: str = "CE") -> str:
    return f"contratos_{uf.lower()}_{data_inicial:%Y%m%d}_{data_final:%Y%m%d}.csv"


def baixar_contratos(
    data_inicial: date,
    data_final: date,
    uf: str = "CE",
    progresso: Optional[Progresso] = None,
) -> tuple[pd.DataFrame, list[int]]:
    """
    Baixa todas as páginas do período e filtra pela UF.
    Retorna (DataFrame, páginas que falharam).
    """
    if data_inicial > data_final:
        raise ValueError("A data inicial não pode ser maior que a data final.")

    def avisar(fracao: float, texto: str) -> None:
        if progresso:
            progresso(fracao, texto)

    params_base = {
        "dataInicial": f"{data_inicial:%Y%m%d}",
        "dataFinal": f"{data_final:%Y%m%d}",
        "tamanhoPagina": TAMANHO_PAGINA,
    }

    avisar(0.0, "Consultando o PNCP...")
    falhas: list[int] = []

    with requests.Session() as sessao:
        primeira = _consultar_pagina(sessao, {**params_base, "pagina": 1})
        if primeira is None:
            raise ErroPNCP(
                "Não foi possível consultar o PNCP agora. Tente novamente em instantes."
            )

        total = int(primeira.get("totalPaginas") or 1)
        registros = list(primeira.get("data") or [])
        avisar(1 / total, f"Baixando página 1 de {total}")

        if total > 1:
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
                futuros = {
                    pool.submit(_consultar_pagina, sessao, {**params_base, "pagina": p}): p
                    for p in range(2, total + 1)
                }
                for concluidas, futuro in enumerate(as_completed(futuros), start=2):
                    resposta = futuro.result()
                    if resposta is None:
                        falhas.append(futuros[futuro])
                    else:
                        registros.extend(resposta.get("data") or [])
                    avisar(concluidas / total, f"Baixando página {concluidas} de {total}")

    df = pd.json_normalize(registros)

    if "unidadeOrgao.ufSigla" in df.columns:
        df = df[df["unidadeOrgao.ufSigla"] == uf]
    if "numeroControlePNCP" in df.columns:
        df = df.drop_duplicates(subset="numeroControlePNCP")

    return df.reset_index(drop=True), sorted(falhas)


def salvar_csv(df: pd.DataFrame, data_inicial: date, data_final: date, uf: str = "CE") -> Path:
    CONTRATOS_DIR.mkdir(parents=True, exist_ok=True)
    destino = CONTRATOS_DIR / nome_arquivo(data_inicial, data_final, uf)
    df.to_csv(destino, index=False, encoding="utf-8-sig")
    return destino


def listar_arquivos() -> list[Path]:
    """CSVs já baixados, do mais recente para o mais antigo."""
    if not CONTRATOS_DIR.exists():
        return []
    return sorted(CONTRATOS_DIR.glob("contratos_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
